# Barbell-only per-exercise TCN phase sanity check

??:

1. YOLO-World barbell trajectory? `(start, middle, finish)` ??? ???? ????.
2. ???(`squat`, `benchpress`, `deadlift`)? **barbell ???** ?? phase-only TCN? ????.
3. ?? deadlift? middle frame? `bottom`? ??? `top` turning point?? ??, `bar_direction` ?? ???? deadlift? `down/up`? swap?? ????.

?? ??:

- per-exercise TCN? ? ???: barbell trajectory ???? phase ??? ??.
- deadlift? ? ???: deadlift ??/trajectory alignment ?? ?? ?? ?? ???? ?.
- per-exercise? ? ???? single model? ? ???: ??? ?? semantics ?? ??? exercise one-hot/embedding? ???.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import random
import time
from dataclasses import dataclass
from typing import Iterable, Optional

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

# Repo import
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model" / "train_ablation.py").exists():
    # If the notebook is opened from a different cwd, walk up a few levels.
    for parent in Path.cwd().resolve().parents:
        if (parent / "model" / "train_ablation.py").exists():
            PROJECT_ROOT = parent
            break

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from model import train_ablation as ta

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)

PROJECT_ROOT: C:\Users\kojy1\PycharmProjects\capstone
DEVICE: cuda


In [2]:
# -----------------------------
# Config
# -----------------------------

EXERCISES = ["squat", "benchpress", "deadlift"]

# label_scheme:
# - "as_labeled": repo ??. ?? ???? start->middle=down, middle->finish=up.
# - "bar_direction": deadlift? start->middle=up, middle->finish=down?? swap.
#   squat/benchpress? as_labeled? ??.
PRIMARY_LABEL_SCHEME = "bar_direction"
COMPARE_DEADLIFT_AS_LABELED = True

CLIP_LEN = 16
STRIDE = 2
BATCH_SIZE = 256
EPOCHS = 40
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN = 64
DROPOUT = 0.20

# ?? ?? ???. ?? ?? ? notebook? ? ??? ? ?? True.
FAST_DEV_RUN = False
MAX_VIDEOS_PER_SPLIT_TYPE = 4 if FAST_DEV_RUN else None
MAX_EPOCHS_IF_FAST = 2

# Feature set:
# center_velocity = normalized barbell center x,y + confidence + dx,dy
# center_velocity_detected = center_velocity + detected mask
FEATURE_MODE = "center_velocity"

OUT_DIR = PROJECT_ROOT / "phase_experiments" / "barbell_yolo_world" / "per_exercise_tcn_probe"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

OUT_DIR: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\barbell_yolo_world\per_exercise_tcn_probe


In [3]:
def find_data_root(project_root: Path) -> Path:
    """Find the dataset root that contains labels/ and STGCN_LSTM/barbell_yolo_world/.

    This avoids hardcoding the Korean directory name because some Windows consoles
    show it through a different codepage.
    """
    candidates = []
    data_dir = project_root / "data"
    if data_dir.exists():
        for label_dir in data_dir.rglob("labels"):
            if (label_dir / "labels.csv").exists():
                root = label_dir.parent
                score = int((root / "STGCN_LSTM" / "barbell_yolo_world").exists())
                candidates.append((score, root))
    if not candidates:
        raise FileNotFoundError("Could not find data root containing labels/labels.csv under ./data")
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

DATA_ROOT = find_data_root(PROJECT_ROOT)
LABEL_DIR = DATA_ROOT / "labels"
VIDEO_DIR = DATA_ROOT
BARBELL_DIR = DATA_ROOT / "STGCN_LSTM" / "barbell_yolo_world"
TEAM_META_CSV = DATA_ROOT / "workdir" / "meta_v4.csv"

print("DATA_ROOT   :", DATA_ROOT)
print("LABEL_DIR   :", LABEL_DIR, LABEL_DIR.exists())
print("BARBELL_DIR :", BARBELL_DIR, BARBELL_DIR.exists())
print("TEAM_META   :", TEAM_META_CSV, TEAM_META_CSV.exists())

labels = ta.load_labels(LABEL_DIR)
print("labels:", len(labels))

DATA_ROOT   : C:\Users\kojy1\PycharmProjects\capstone\data\측면\data
LABEL_DIR   : C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\labels True
BARBELL_DIR : C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\STGCN_LSTM\barbell_yolo_world True
TEAM_META   : C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\workdir\meta_v4.csv False
labels: 210


## 1) Build metadata using existing repo split/cache contract

`train_ablation.build_meta_df()`? ??? ?? ?? split? barbell cache ??? ???.

In [4]:
meta = ta.build_meta_df(
    labels,
    video_dir=VIDEO_DIR,
    barbell_dir=BARBELL_DIR,
    pose_backend=ta.POSE_BACKEND_BARBELL,
    barbell_missing_policy="error",
    team_meta_csv=TEAM_META_CSV,
    use_team_split=True,
    seed=SEED,
)

if MAX_VIDEOS_PER_SPLIT_TYPE:
    meta = ta.limit_meta_per_split_type(meta, MAX_VIDEOS_PER_SPLIT_TYPE)

meta = meta.reset_index(drop=True)
display(meta.groupby(["split", "type"]).size().unstack(fill_value=0).reindex(["train", "val"]))
display(meta.head())

type,benchpress,deadlift,squat
split,,,
train,75,20,71
val,19,5,18


,type,name,cls,T,fps,n_reps,has_incomplete,npz,video_path,pose_backend,num_kpt,source_num_kpt,pose_graph_id,source_graph_id,split
0,squat,squat_2.MOV,0,430,29.972119,4,False,C:\Users\kojy1\PycharmProjects\capstone\data\측...,C:\Users\kojy1\PycharmProjects\capstone\data\측...,barbell,1,1,barbell1,barbell1,train
1,squat,squat_3.MOV,0,639,29.974201,5,False,C:\Users\kojy1\PycharmProjects\capstone\data\측...,C:\Users\kojy1\PycharmProjects\capstone\data\측...,barbell,1,1,barbell1,barbell1,val
2,squat,squat_4.MOV,0,600,29.972525,4,False,C:\Users\kojy1\PycharmProjects\capstone\data\측...,C:\Users\kojy1\PycharmProjects\capstone\data\측...,barbell,1,1,barbell1,barbell1,val
3,squat,squat_5.MOV,0,656,29.972586,4,False,C:\Users\kojy1\PycharmProjects\capstone\data\측...,C:\Users\kojy1\PycharmProjects\capstone\data\측...,barbell,1,1,barbell1,barbell1,train
4,squat,squat_9.mp4,0,72,23.976024,1,False,C:\Users\kojy1\PycharmProjects\capstone\data\측...,C:\Users\kojy1\PycharmProjects\capstone\data\측...,barbell,1,1,barbell1,barbell1,train


## 2) Turning-point direction probe

??? ????? y? ?? `0=?`, `1=??`?.

- squat/benchpress? middle? ????? `y_middle > y_start` ??? `y_middle > y_finish` ??.
- deadlift? middle? ????? `y_middle < y_start` ??? `y_middle < y_finish` ??.

In [5]:
def load_barbell_arrays(npz_path: str | Path) -> dict[str, np.ndarray]:
    with np.load(npz_path) as d:
        kpts = d["kpts"].astype(np.float32)
        centers = d["centers"].astype(np.float32) if "centers" in d.files else kpts[:, 0, :2].astype(np.float32)
        conf = d["detection_conf"].astype(np.float32) if "detection_conf" in d.files else kpts[:, 0, 2].astype(np.float32)
        detected = d["detected"].astype(bool) if "detected" in d.files else (conf > 0)
        boxes = d["boxes"].astype(np.float32) if "boxes" in d.files else None
    return {"kpts": kpts, "centers": centers, "conf": conf, "detected": detected, "boxes": boxes}


def turning_probe(meta_df: pd.DataFrame, labels: dict) -> pd.DataFrame:
    rows = []
    for _, r in meta_df.iterrows():
        typ, name = r["type"], r["name"]
        arr = load_barbell_arrays(r["npz"])
        y = arr["centers"][:, 1]
        T = len(y)
        for rep_idx, (s, m, f) in enumerate(labels[(typ, name)]["reps"], start=1):
            if T <= 0:
                continue
            s = max(0, min(int(s), T - 1))
            m = max(0, min(int(m), T - 1))
            f = max(0, min(int(f), T - 1))
            ys, ym, yf = float(y[s]), float(y[m]), float(y[f])
            rows.append({
                "type": typ,
                "name": name,
                "split": r["split"],
                "rep": rep_idx,
                "start": s,
                "middle": m,
                "finish": f,
                "y_start": ys,
                "y_middle": ym,
                "y_finish": yf,
                "middle_is_top": bool(ym < min(ys, yf)),
                "middle_is_bottom": bool(ym > max(ys, yf)),
                "amp_first": ym - ys,
                "amp_second": yf - ym,
            })
    return pd.DataFrame(rows)

turn_df = turning_probe(meta, labels)
summary = turn_df.groupby("type").agg(
    reps=("rep", "count"),
    middle_top_frac=("middle_is_top", "mean"),
    middle_bottom_frac=("middle_is_bottom", "mean"),
    median_amp_first=("amp_first", "median"),
    median_amp_second=("amp_second", "median"),
)
print("Turning point orientation summary")
display(summary.round(4))

display(turn_df.sort_values(["type", "name", "rep"]).head(20))

turn_df.to_csv(OUT_DIR / "turning_point_direction_probe.csv", index=False, encoding="utf-8-sig")
print("saved:", OUT_DIR / "turning_point_direction_probe.csv")

Turning point orientation summary


,reps,middle_top_frac,middle_bottom_frac,median_amp_first,median_amp_second
type,,,,,
benchpress,578,0.1142,0.8097,0.1219,-0.1199
deadlift,53,0.8491,0.0755,-0.1319,0.1359
squat,857,0.0782,0.8203,0.1218,-0.1222


,type,name,split,rep,start,middle,finish,y_start,y_middle,y_finish,middle_is_top,middle_is_bottom,amp_first,amp_second
857,benchpress,bench press_1.mp4,val,1,0,29,49,0.604984,0.761217,0.616634,False,True,0.156233,-0.144582
858,benchpress,bench press_10.mp4,train,1,0,34,52,0.305375,0.530673,0.285287,False,True,0.225298,-0.245386
859,benchpress,bench press_11.mp4,train,1,0,34,38,0.288125,0.496554,0.301576,False,True,0.208430,-0.194978
860,benchpress,bench press_12.mp4,train,1,0,95,146,0.514610,0.544812,0.508364,False,True,0.030202,-0.036448
861,benchpress,bench press_14.mp4,val,1,0,34,50,0.535507,0.631488,0.524888,False,True,0.095981,-0.106600
862,benchpress,bench press_16.mp4,train,1,0,50,71,0.366761,0.506864,0.334801,False,True,0.140103,-0.172063
863,benchpress,bench press_19.mp4,train,1,0,43,70,0.284374,0.653224,0.233474,False,True,0.368850,-0.419750
864,benchpress,bench press_2.mp4,val,1,0,29,49,0.609372,0.668622,0.649638,False,True,0.059250,-0.018984
865,benchpress,bench press_20.mp4,val,1,0,43,64,0.224130,0.633001,0.391744,False,True,0.408871,-0.241257
866,benchpress,bench press_22.mp4,train,1,0,29,61,0.521329,0.531305,0.526722,False,True,0.009976,-0.004583


saved: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\barbell_yolo_world\per_exercise_tcn_probe\turning_point_direction_probe.csv


## 3) Phase label schemes

- `as_labeled`: ?? repo ??. ?? ???? `start:middle=down`, `middle:finish=up`.
- `bar_direction`: deadlift? ?? ?? ??? ?? `start:middle=up`, `middle:finish=down`?? ???.

??: per-exercise ????? class id swap??? macro-F1? ?? ??? ?? ? ??. ? swap? ??? single model? ?? ? semantics ??? ??? ?? ????.

In [6]:
PHASE_READY = 0
PHASE_DOWN = 1
PHASE_UP = 2
PHASE_NAMES = ["ready", "down", "up"]


def make_phase_target_for_scheme(
    T: int,
    reps: Iterable[tuple[int, int, int]],
    exercise: str,
    scheme: str = "bar_direction",
    dtype=np.int64,
) -> np.ndarray:
    phase = np.zeros(int(T), dtype=dtype)
    if T <= 0:
        return phase
    scheme = str(scheme)
    for s, m, f in reps:
        s = max(0, min(int(s), T - 1))
        m = max(0, min(int(m), T - 1))
        f = max(0, min(int(f), T - 1))
        if scheme == "as_labeled":
            first_label, second_label = PHASE_DOWN, PHASE_UP
        elif scheme == "bar_direction":
            if exercise == "deadlift":
                first_label, second_label = PHASE_UP, PHASE_DOWN
            else:
                first_label, second_label = PHASE_DOWN, PHASE_UP
        else:
            raise ValueError(f"unknown label scheme: {scheme}")
        if m > s:
            phase[s:m] = first_label
        if f > m:
            phase[m:f] = second_label
    return phase


def phase_fraction_table(meta_df: pd.DataFrame, scheme: str) -> pd.DataFrame:
    rows = []
    for _, r in meta_df.iterrows():
        typ, name = r["type"], r["name"]
        T = int(r["T"])
        phases = make_phase_target_for_scheme(T, labels[(typ, name)]["reps"], typ, scheme)
        rows.append({
            "type": typ,
            "name": name,
            "split": r["split"],
            "ready_frac": float((phases == PHASE_READY).mean()),
            "down_frac": float((phases == PHASE_DOWN).mean()),
            "up_frac": float((phases == PHASE_UP).mean()),
        })
    return pd.DataFrame(rows)

for scheme in ["as_labeled", "bar_direction"]:
    print("\nSCHEME:", scheme)
    display(phase_fraction_table(meta, scheme).groupby(["split", "type"])[["ready_frac", "down_frac", "up_frac"]].mean().round(4))


SCHEME: as_labeled


ready_frac  down_frac  up_frac
split type                                      
train benchpress      0.3628     0.3474   0.2899
      deadlift        0.4329     0.3254   0.2417
      squat           0.3245     0.3553   0.3202
val   benchpress      0.2868     0.3945   0.3187
      deadlift        0.4386     0.3252   0.2362
      squat           0.3733     0.3249   0.3017


SCHEME: bar_direction


ready_frac  down_frac  up_frac
split type                                      
train benchpress      0.3628     0.3474   0.2899
      deadlift        0.4329     0.2417   0.3254
      squat           0.3245     0.3553   0.3202
val   benchpress      0.2868     0.3945   0.3187
      deadlift        0.4386     0.2362   0.3252
      squat           0.3733     0.3249   0.3017

## 4) Dataset: barbell center + velocity only

?? feature ???:

```text
[x, y, conf, dx, dy]
```

- `x,y`: frame-normalized center? `[-1, 1]`? ??.
- `dx,dy`: normalized coordinate? 1-frame difference.
- label: clip ? frame? phase.

In [7]:
def build_barbell_features(npz_path: str | Path, feature_mode: str = FEATURE_MODE) -> np.ndarray:
    arr = load_barbell_arrays(npz_path)
    kpts = arr["kpts"]
    T = len(kpts)
    if T == 0:
        return np.zeros((0, 5), dtype=np.float32)

    # kpts already contains smoothed center x/y and detection confidence.
    xy = kpts[:, 0, :2].astype(np.float32)
    xy = (xy - 0.5) * 2.0
    conf = np.nan_to_num(kpts[:, 0, 2:3].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    conf = np.clip(conf, 0.0, 1.0)

    dxy = np.zeros_like(xy, dtype=np.float32)
    if T > 1:
        dxy[1:] = xy[1:] - xy[:-1]

    feats = [xy, conf, dxy]
    if feature_mode == "center_velocity_detected":
        detected = arr["detected"].astype(np.float32).reshape(-1, 1)
        feats.append(detected)
    elif feature_mode != "center_velocity":
        raise ValueError(f"unknown feature_mode={feature_mode}")

    return np.concatenate(feats, axis=1).astype(np.float32)


class BarbellPhaseWindowDataset(Dataset):
    def __init__(
        self,
        meta_df: pd.DataFrame,
        labels: dict,
        exercise: str,
        split: str,
        label_scheme: str,
        clip_len: int = CLIP_LEN,
        stride: int = STRIDE,
        feature_mode: str = FEATURE_MODE,
    ):
        self.exercise = exercise
        self.split = split
        self.label_scheme = label_scheme
        self.clip_len = int(clip_len)
        self.stride = int(stride)
        sub = meta_df[(meta_df["type"] == exercise) & (meta_df["split"] == split)].reset_index(drop=True)
        self.videos = []
        self.samples = []
        for _, r in sub.iterrows():
            typ, name = r["type"], r["name"]
            feats = build_barbell_features(r["npz"], feature_mode=feature_mode)
            T = min(int(r["T"]), len(feats))
            if T <= 0:
                continue
            feats = feats[:T]
            phases = make_phase_target_for_scheme(T, labels[(typ, name)]["reps"], typ, label_scheme)
            vid_idx = len(self.videos)
            self.videos.append({
                "type": typ,
                "name": name,
                "npz": r["npz"],
                "T": T,
                "features": feats,
                "phases": phases,
            })
            if T < self.clip_len:
                ends = [T - 1]
            else:
                ends = list(range(self.clip_len - 1, T, self.stride))
                if ends[-1] != T - 1:
                    ends.append(T - 1)
            for end in ends:
                self.samples.append((vid_idx, int(end)))
        self.input_channels = int(self.videos[0]["features"].shape[1]) if self.videos else 0

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        vid_idx, end = self.samples[idx]
        v = self.videos[vid_idx]
        feats = v["features"]
        phase = v["phases"]
        start = end - self.clip_len + 1
        if start >= 0:
            clip = feats[start:end + 1]
        else:
            pad = -start
            clip = np.concatenate([np.tile(feats[0:1], (pad, 1)), feats[:end + 1]], axis=0)
        assert clip.shape[0] == self.clip_len
        x = torch.from_numpy(clip.T.copy()).float()  # [C,T]
        y = torch.tensor(int(phase[end]), dtype=torch.long)
        return x, y

# Dataset smoke check
for ex in EXERCISES:
    ds = BarbellPhaseWindowDataset(meta, labels, ex, "train", PRIMARY_LABEL_SCHEME)
    print(ex, "videos", len(ds.videos), "samples", len(ds), "input_channels", ds.input_channels)
    if len(ds):
        x, y = ds[0]
        print("  sample", tuple(x.shape), y.item(), PHASE_NAMES[y.item()])

squat videos 71 samples 29050 input_channels 5
  sample (5, 16) 1 down
benchpress videos 75 samples 24951 input_channels 5
  sample (5, 16) 1 down
deadlift videos 20 samples 2321 input_channels 5
  sample (5, 16) 0 ready


## 5) TCN model + training utilities

TCN? causal Conv1d stack? ?? ??? timestep feature? clip ? frame phase? ????.

In [8]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=self.pad, dilation=dilation)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.conv(x)
        if self.pad > 0:
            y = y[..., :-self.pad]
        return y


class TCNBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, dilation: int = 1, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, kernel_size=kernel_size, dilation=dilation),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.res = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x) + self.res(x)


class BarbellPhaseTCN(nn.Module):
    def __init__(self, in_ch: int, hidden: int = HIDDEN, num_phase: int = 3, dropout: float = DROPOUT):
        super().__init__()
        self.tcn = nn.Sequential(
            TCNBlock(in_ch, hidden, dilation=1, dropout=dropout),
            TCNBlock(hidden, hidden, dilation=2, dropout=dropout),
            TCNBlock(hidden, hidden, dilation=4, dropout=dropout),
            TCNBlock(hidden, hidden, dilation=8, dropout=dropout),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_phase),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,C,T]
        z = self.tcn(x)
        last = z[:, :, -1]
        return self.head(last)


def class_weights_from_dataset(ds: Dataset, num_classes: int = 3) -> torch.Tensor:
    counts = np.zeros(num_classes, dtype=np.float64)
    for _, y in ds:
        counts[int(y)] += 1
    freq = counts / max(counts.sum(), 1)
    weights = 1.0 / (freq + 1e-6)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


@torch.no_grad()
def eval_loader(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    y_true, y_pred = [], []
    probs = []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        prob = logits.softmax(dim=1).cpu().numpy()
        pred = prob.argmax(axis=1)
        y_true.extend(y.numpy().astype(int).tolist())
        y_pred.extend(pred.astype(int).tolist())
        probs.append(prob)
    if not y_true:
        return {"acc": np.nan, "macro_f1": np.nan, "per_phase_f1": [np.nan] * 3, "n": 0}
    _, _, per_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0
    )
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_phase_f1": [float(v) for v in per_f1],
        "support": [int(v) for v in support],
        "n": int(len(y_true)),
        "confusion": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
    }


def train_one_exercise(
    exercise: str,
    label_scheme: str = PRIMARY_LABEL_SCHEME,
    epochs: int = EPOCHS,
    patience: int = PATIENCE,
) -> dict:
    train_ds = BarbellPhaseWindowDataset(meta, labels, exercise, "train", label_scheme)
    val_ds = BarbellPhaseWindowDataset(meta, labels, exercise, "val", label_scheme)
    if len(train_ds) == 0 or len(val_ds) == 0:
        raise RuntimeError(f"empty dataset for {exercise}: train={len(train_ds)} val={len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    model = BarbellPhaseTCN(in_ch=train_ds.input_channels).to(DEVICE)
    weights = class_weights_from_dataset(train_ds).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best = {"macro_f1": -1.0, "epoch": 0, "state": None, "metrics": None}
    history = []
    bad = 0
    use_epochs = min(epochs, MAX_EPOCHS_IF_FAST) if FAST_DEV_RUN else epochs
    start_time = time.time()

    for ep in range(1, use_epochs + 1):
        model.train()
        losses = []
        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(float(loss.item()))

        vm = eval_loader(model, val_loader)
        row = {"epoch": ep, "train_loss": float(np.mean(losses)), **{f"val_{k}": v for k, v in vm.items() if k != "confusion"}}
        history.append(row)
        print(
            f"[{exercise}/{label_scheme}] ep={ep:03d} "
            f"loss={row['train_loss']:.4f} val_acc={vm['acc']:.4f} val_f1={vm['macro_f1']:.4f} "
            f"per_f1={[round(v,4) for v in vm['per_phase_f1']]}"
        )

        if vm["macro_f1"] > best["macro_f1"]:
            best = {
                "macro_f1": vm["macro_f1"],
                "epoch": ep,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "metrics": vm,
            }
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop at ep={ep}, best_ep={best['epoch']}")
                break

    elapsed = time.time() - start_time
    if best["state"] is not None:
        model.load_state_dict(best["state"])

    result = {
        "exercise": exercise,
        "label_scheme": label_scheme,
        "feature_mode": FEATURE_MODE,
        "clip_len": CLIP_LEN,
        "stride": STRIDE,
        "epochs_run": len(history),
        "best_epoch": int(best["epoch"]),
        "elapsed_sec": float(elapsed),
        "train_videos": len(train_ds.videos),
        "val_videos": len(val_ds.videos),
        "train_samples": len(train_ds),
        "val_samples": len(val_ds),
        "input_channels": train_ds.input_channels,
        "val_acc": float(best["metrics"]["acc"]),
        "val_macro_f1": float(best["metrics"]["macro_f1"]),
        "ready_f1": float(best["metrics"]["per_phase_f1"][0]),
        "down_f1": float(best["metrics"]["per_phase_f1"][1]),
        "up_f1": float(best["metrics"]["per_phase_f1"][2]),
        "support_ready": int(best["metrics"]["support"][0]),
        "support_down": int(best["metrics"]["support"][1]),
        "support_up": int(best["metrics"]["support"][2]),
        "confusion": best["metrics"]["confusion"].tolist(),
        "history": history,
    }

    # Save lightweight artifacts per run.
    run_dir = OUT_DIR / f"{exercise}_{label_scheme}_{FEATURE_MODE}_c{CLIP_LEN}"
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "result": result}, run_dir / "best.pt")
    (run_dir / "result.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False, encoding="utf-8-sig")
    return result

## 6) Run per-exercise sanity check

?? ??:

- squat: `bar_direction`
- benchpress: `bar_direction`
- deadlift: `bar_direction`
- deadlift ?? ??: `as_labeled`

`deadlift`?? ? scheme? macro-F1? ??? ? ??. per-exercise??? class id? swap? ???, ??? ? **deadlift TCN ??? ?????**?.

In [9]:
runs = [(ex, PRIMARY_LABEL_SCHEME) for ex in EXERCISES]
if COMPARE_DEADLIFT_AS_LABELED and PRIMARY_LABEL_SCHEME != "as_labeled":
    runs.append(("deadlift", "as_labeled"))

results = []
for exercise, scheme in runs:
    print("\n" + "=" * 80)
    print("RUN", exercise, scheme)
    result = train_one_exercise(exercise, scheme)
    results.append(result)

summary_df = pd.DataFrame([{k: v for k, v in r.items() if k not in {"history", "confusion"}} for r in results])
display(summary_df[[
    "exercise", "label_scheme", "val_acc", "val_macro_f1", "ready_f1", "down_f1", "up_f1",
    "best_epoch", "train_videos", "val_videos", "train_samples", "val_samples", "input_channels"
]].round(4))

summary_path = OUT_DIR / "per_exercise_tcn_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("saved:", summary_path)


RUN squat bar_direction
[squat/bar_direction] ep=001 loss=0.9211 val_acc=0.5477 val_f1=0.5419 per_f1=[0.5584, 0.4585, 0.6089]
[squat/bar_direction] ep=002 loss=0.8307 val_acc=0.5430 val_f1=0.5367 per_f1=[0.5336, 0.4657, 0.6107]
[squat/bar_direction] ep=003 loss=0.7876 val_acc=0.5387 val_f1=0.5329 per_f1=[0.5142, 0.4715, 0.613]
[squat/bar_direction] ep=004 loss=0.7609 val_acc=0.5590 val_f1=0.5546 per_f1=[0.5279, 0.5105, 0.6254]
[squat/bar_direction] ep=005 loss=0.7429 val_acc=0.5606 val_f1=0.5567 per_f1=[0.539, 0.4945, 0.6367]
[squat/bar_direction] ep=006 loss=0.7283 val_acc=0.5604 val_f1=0.5559 per_f1=[0.5437, 0.4814, 0.6428]
[squat/bar_direction] ep=007 loss=0.7203 val_acc=0.5533 val_f1=0.5421 per_f1=[0.5284, 0.4514, 0.6465]
[squat/bar_direction] ep=008 loss=0.7096 val_acc=0.5747 val_f1=0.5709 per_f1=[0.5533, 0.5154, 0.6441]
[squat/bar_direction] ep=009 loss=0.7006 val_acc=0.5613 val_f1=0.5548 per_f1=[0.5288, 0.4834, 0.6523]
[squat/bar_direction] ep=010 loss=0.6910 val_acc=0.5749 val

,exercise,label_scheme,val_acc,val_macro_f1,ready_f1,down_f1,up_f1,best_epoch,train_videos,val_videos,train_samples,val_samples,input_channels
0,squat,bar_direction,0.5883,0.5842,0.5570,0.5320,0.6635,39,71,18,29050,7132,5
1,benchpress,bar_direction,0.4694,0.4714,0.4343,0.5043,0.4756,17,75,19,24951,3813,5
2,deadlift,bar_direction,0.5391,0.5375,0.6097,0.5635,0.4394,22,20,5,2321,755,5
3,deadlift,as_labeled,0.4940,0.4705,0.6216,0.3121,0.4778,9,20,5,2321,755,5


saved: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\barbell_yolo_world\per_exercise_tcn_probe\per_exercise_tcn_summary.csv


## 7) Read result

?? ???:

```text
squat / bench / deadlift ?? macro-F1? ?? barbell ST-GCN?? ?? ??
=> ?? ?????? phase ??? ??. ?? temporal_avg/ST-GCN ??? ???? ???.

squat/bench? ??, deadlift? ??
=> deadlift ?? ?? ?? barbell ??/trajectory ?? ??? ?? ?? ?.

per-exercise? ????? single model? ??
=> ??? ?? semantics ??. single model?? exercise one-hot/embedding ??.
```

?? ?? ??:

```text
barbell ST-GCN full:
  squat      macro-F1 ? 0.560
  benchpress macro-F1 ? 0.472
  deadlift   macro-F1 ? 0.174
```

In [10]:
# Optional: load saved summary later without rerunning training.
summary_path = OUT_DIR / "per_exercise_tcn_summary.csv"
if summary_path.exists():
    loaded = pd.read_csv(summary_path)
    display(loaded[[
        "exercise", "label_scheme", "val_acc", "val_macro_f1", "ready_f1", "down_f1", "up_f1",
        "best_epoch", "train_videos", "val_videos", "train_samples", "val_samples"
    ]].round(4))
else:
    print("No saved summary yet:", summary_path)

,exercise,label_scheme,val_acc,val_macro_f1,ready_f1,down_f1,up_f1,best_epoch,train_videos,val_videos,train_samples,val_samples
0,squat,bar_direction,0.5883,0.5842,0.5570,0.5320,0.6635,39,71,18,29050,7132
1,benchpress,bar_direction,0.4694,0.4714,0.4343,0.5043,0.4756,17,75,19,24951,3813
2,deadlift,bar_direction,0.5391,0.5375,0.6097,0.5635,0.4394,22,20,5,2321,755
3,deadlift,as_labeled,0.4940,0.4705,0.6216,0.3121,0.4778,9,20,5,2321,755
